# Should Retail-Core's protect list rank members by how often they ordered in Q2, instead of by how much they spent?

**Week 2, Wednesday. The second case, in pairs or alone, in the take-home.**

> "Frequency is what fell in Retail-Plus. Before it spreads, I want Retail-Core's top fifty
> ranked by how often members ordered in Q2. Same rule as the head of Retail-Plus: ties ranked
> the same, and tell me how many made it."
> The marketing lead, Kalpa Retail

**The situation.** Retail-Core is Kalpa Retail's everyday shoppers, 96 of whom ordered in Q2
(July to September 2026). Ranked by Q2 revenue, booked revenue of every order whatever its
status, Retail-Core's top fifty under RANK holds 50 members, because no two members share the
fiftieth figure. Frequency is how many orders a member placed in the quarter. Under RANK,
members who spent, or ordered, the same share a place, and the list says how many made it.

**The data.** Kalpa's Postgres warehouse: `orders` (1,000 rows: order_id, customer_id,
order_date, quarter, channel, amount, status) and `customers` (340 rows, one per member, with
the segment).

**How to answer.** Seven `TODO` markers, each a choice lettered a to d. Replace each
placeholder with the letter in quotes, for example `"a"`, and run the cell; the check cell
after it tells you whether your step behaves. Post your seven letters in order, then answer
the brief's three design items together.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit

import re
from decimal import Decimal
SQL_DIR = kit.data_dir().parent / "sql"

def blocks(stem):
    """Every named block of a .sql file in ../sql/, keyed by the name on its '-- name:' line."""
    text = (SQL_DIR / f"C2_W02_D03_{stem}_STUDENT.sql").read_text(encoding="utf-8")
    parts = re.split(r"^-- name: (\w+)\s*$", text, flags=re.M)
    return {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}

def num(v):
    """A database number as a Python int or float."""
    if isinstance(v, Decimal):
        return int(v) if v == v.to_integral_value() else float(v)
    return v

def rows(query):
    """Run a query and hand back its rows with every number as a Python number."""
    return [{k: num(v) for k, v in r.items()} for r in kit.sql(query)]

def show(found, caption="", money=(), limit=12):
    """Rows as the kit's table, with the columns named in money set in rupees."""
    if not found:
        kit.table(["result"], [["no rows"]], caption)
        return
    heads = list(found[0])
    def cell(h, v):
        v = num(v)
        if v is None:
            return "NULL"
        if h in money:
            return kit.rupees(v)
        return f"{v:,}" if isinstance(v, int) and abs(v) >= 10000 else str(v)
    body = [[cell(h, r[h]) for h in heads] for r in found[:limit]]
    if len(found) > limit:
        body.append(["..." for _ in heads])
    kit.table(heads, body, caption)

def run(name, caption="", money=(), limit=12, echo=True):
    """Print a named block, run it, show its rows and hand them back."""
    if echo:
        print(Q[name])
    found = rows(Q[name])
    show(found, caption, money, limit)
    return found

def lakh(v):
    """Rupees in lakh, for an axis that has to hold Business and retail members at once."""
    return f"{v / 100000:,.1f} lakh"

Q = {}
print("connected:", kit.sql("select version()")[0]["version"].split(",")[0])

Q2_SPEND = """
    SELECT c.segment, o.customer_id, count(*) AS q2_orders, sum(o.amount) AS q2_revenue
    FROM   orders o
    JOIN   customers c USING (customer_id)
    WHERE  o.quarter = 'Q2'
    GROUP  BY c.segment, o.customer_id"""

MONTHLY = """
    SELECT customer_id, date_trunc('month', order_date)::date AS month, sum(amount) AS spend
    FROM   orders
    GROUP  BY customer_id, date_trunc('month', order_date)"""

In [ ]:
kit.flow(["orders per member", "rank by orders", "a second key", "the two lists compared", "the line"], lit=1)

## Step 1. How many Q2 orders did each Retail-Core member place, and where do the ties fall?

Where this is used at work: a frequency metric is a count, and counts tie far more often than
rupee amounts do.

In [ ]:
# TODO 1. Which expression counts a member's Q2 orders?
#   a) count(*), one per order row the member placed
#   b) count(DISTINCT o.customer_id)
#   c) sum(o.amount)
#   d) count(DISTINCT date_trunc('month', o.order_date))
ORDERS = {"a": "count(*)", "b": "count(DISTINCT o.customer_id)", "c": "sum(o.amount)",
          "d": "count(DISTINCT date_trunc('month', o.order_date))"}[__TODO1__]
core = rows(f"""SELECT o.customer_id, {ORDERS} AS q2_orders, sum(o.amount) AS q2_revenue
                FROM orders o JOIN customers c USING (customer_id)
                WHERE o.quarter = 'Q2' AND c.segment = 'Retail-Core'
                GROUP BY o.customer_id""")
spread = {}
for r in core:
    spread[r["q2_orders"]] = spread.get(r["q2_orders"], 0) + 1
kit.columns([str(k) for k in sorted(spread, reverse=True)], [("members", [spread[k] for k in sorted(spread, reverse=True)])],
            title="Retail-Core members by their number of Q2 orders")

In [ ]:
kit.check("96 Retail-Core members ordered in Q2", len(core) == 96)
kit.check("the order counts add back to the segment's Q2 orders",
          sum(r["q2_orders"] for r in core) == rows("SELECT count(*) AS n FROM orders o JOIN customers c USING (customer_id) "
                                                    "WHERE o.quarter = 'Q2' AND c.segment = 'Retail-Core'")[0]["n"])

## Step 2. How many members does each rule ship when the list is ranked by orders alone?

Where this is used at work: a tie rule chosen for rupee amounts has to be read again when the
metric becomes a count.

In [ ]:
# TODO 2. Which window gives members with the same number of orders the same place, and skips the places they use up?
#   a) row_number() OVER (ORDER BY q2_orders DESC, customer_id)
#   b) rank() OVER (ORDER BY q2_orders DESC)
#   c) dense_rank() OVER (ORDER BY q2_orders DESC)
#   d) rank() OVER (ORDER BY q2_revenue DESC)
RULE = {"a": "row_number() OVER (ORDER BY q2_orders DESC, customer_id)", "b": "rank() OVER (ORDER BY q2_orders DESC)",
        "c": "dense_rank() OVER (ORDER BY q2_orders DESC)", "d": "rank() OVER (ORDER BY q2_revenue DESC)"}[__TODO2__]
base = f"""SELECT o.customer_id, count(*) AS q2_orders, sum(o.amount) AS q2_revenue
           FROM orders o JOIN customers c USING (customer_id)
           WHERE o.quarter = 'Q2' AND c.segment = 'Retail-Core' GROUP BY o.customer_id"""
ranked = rows(f"SELECT customer_id, q2_orders, q2_revenue, {RULE} AS place FROM ({base}) b")
chosen = sum(1 for r in ranked if r["place"] <= 50)
all_rules = rows(f"""SELECT count(*) FILTER (WHERE rn <= 50) AS row_number_ships, count(*) FILTER (WHERE rk <= 50) AS rank_ships,
                            count(*) FILTER (WHERE dr <= 50) AS dense_rank_ships
                     FROM (SELECT row_number() OVER (ORDER BY q2_orders DESC, customer_id) AS rn,
                                  rank() OVER (ORDER BY q2_orders DESC) AS rk,
                                  dense_rank() OVER (ORDER BY q2_orders DESC) AS dr FROM ({base}) b) x""")[0]
kit.bars([("row_number", all_rules["row_number_ships"]), ("rank", all_rules["rank_ships"]),
          ("dense_rank", all_rules["dense_rank_ships"]), ("your rule", chosen)],
         title="Rows each rule ships when Retail-Core is ranked by Q2 orders alone", lit=(3,))

In [ ]:
same_orders_same_place = all(len({r["place"] for r in ranked if r["q2_orders"] == n}) == 1 for n in spread)
kit.check("members with the same number of orders share a place under your rule", same_orders_same_place)
kit.check("your rule ships what the head of Retail-Plus's rule ships on these counts", chosen == all_rules["rank_ships"])
kit.check("your rule ranks by orders, so the member with most orders is first",
          min(ranked, key=lambda r: r["place"])["q2_orders"] == max(spread))

## Step 3. Why does that rule ship the number it ships?

Where this is used at work: a stakeholder who asked for fifty and receives more wants the
reason in one sentence.

In [ ]:
# TODO 3. Why does RANK on orders alone ship 51 members?
#   a) 24 members placed three or more orders, and the 27 members with two orders all share 25th place.
#   b) RANK skips a number after every tie, and those skipped numbers count as extra members on the list.
#   c) One member placed eight orders, so RANK counts that member several times on the list.
#   d) The 45 members with one order share a place, and RANK adds one of them to make the list even.
WHY = __TODO3__
print("your reason:", WHY)

## Step 4. Which second key breaks the crowd of ties, and how many members does the list ship then?

Where this is used at work: a ranking on a count almost always needs a second key, and the
second key is a business choice with a reason.

In [ ]:
# TODO 4. Which ORDER BY ranks by orders first, then lets revenue separate members with the same orders?
#   a) ORDER BY q2_orders DESC, customer_id
#   b) ORDER BY q2_orders DESC, q2_revenue DESC
#   c) ORDER BY q2_revenue DESC, q2_orders DESC
#   d) ORDER BY q2_orders DESC
ORDER = {"a": "q2_orders DESC, customer_id", "b": "q2_orders DESC, q2_revenue DESC",
         "c": "q2_revenue DESC, q2_orders DESC", "d": "q2_orders DESC"}[__TODO4__]
freq_list = rows(f"""SELECT customer_id, q2_orders, q2_revenue FROM (
                        SELECT customer_id, q2_orders, q2_revenue, rank() OVER (ORDER BY {ORDER}) AS place
                        FROM ({base}) b) x WHERE place <= 50""")
rev_list = rows(f"""SELECT customer_id, q2_orders, q2_revenue FROM (
                       SELECT customer_id, q2_orders, q2_revenue, rank() OVER (ORDER BY q2_revenue DESC) AS place
                       FROM ({base}) b) x WHERE place <= 50""")
kit.stats([(len(freq_list), "on the frequency list", "RANK on your ORDER BY"),
           (len(rev_list), "on the revenue list", "RANK on Q2 revenue")], caption="Two lists, one rule")

In [ ]:
kit.check("the frequency list puts orders first", max(r["q2_orders"] for r in freq_list) == max(spread)
          and min(r["q2_orders"] for r in freq_list) >= 2)
kit.check("the second key leaves no crowd at the line", len(freq_list) <= 50)
two_kept = [r["q2_revenue"] for r in freq_list if r["q2_orders"] == 2]
two_left = [r["q2_revenue"] for r in core if r["q2_orders"] == 2 and r["customer_id"] not in {f["customer_id"] for f in freq_list}]
kit.check("among members with two orders, the list keeps the ones who spent more",
          not two_left or not two_kept or min(two_kept) >= max(two_left))

## Step 5. How many members do the two lists share, and how far apart are they in rupees?

Where this is used at work: before a team argues over two definitions, it measures how much
the answer actually changes.

In [ ]:
# TODO 5. Which query counts the members who are on both lists?
#   a) An INNER JOIN of the two lists on customer_id, counting the rows it returns
#   b) A LEFT JOIN from the revenue list to the frequency list, counting all the rows it returns
#   c) UNION ALL of the two lists, counting the rows
#   d) The two list counts subtracted, 50 less 50
BOTH = __TODO5__
f_ids, r_ids = {r["customer_id"] for r in freq_list}, {r["customer_id"] for r in rev_list}
shared = {"a": len(f_ids & r_ids), "b": len(r_ids), "c": len(freq_list) + len(rev_list),
          "d": len(freq_list) - len(rev_list)}[BOTH]
only_freq = sorted(f_ids - r_ids)
only_rev = sorted(r_ids - f_ids)
kit.bars([("by orders, then revenue", sum(r["q2_revenue"] for r in freq_list)),
          ("by revenue", sum(r["q2_revenue"] for r in rev_list))],
         title="Q2 revenue each Retail-Core list carries", fmt=kit.rupees)
kit.table(["list", "members", "Q2 revenue carried", "only on this list"],
          [("by orders, then revenue", len(freq_list), kit.rupees(sum(r["q2_revenue"] for r in freq_list)), ", ".join(only_freq)),
           ("by revenue", len(rev_list), kit.rupees(sum(r["q2_revenue"] for r in rev_list)), ", ".join(only_rev))],
          caption=f"Members on both lists, by your query: {shared}")

In [ ]:
kit.check("your count of shared members is the overlap of the two lists", shared == len(f_ids & r_ids))
kit.check("each list has members the other lacks", bool(only_freq) and bool(only_rev))

## Step 6. What do you tell the marketing lead?

Where this is used at work: a definition argument ends when someone says what changes, by how
much, and which choice they recommend.

In [ ]:
# TODO 6. How far apart are the two lists in the Q2 revenue they carry?
#   a) Rs 40
#   b) Rs 2,950
#   c) About Rs 1.2 lakh
#   d) Nothing, since both lists hold fifty members
GAP = __TODO6__

# TODO 7. Which line goes to the marketing lead?
#   a) "Rank Retail-Core by orders alone under RANK: 51 members ship, which honours the tie rule and protects frequency."
#   b) "Rank by orders with Q2 revenue as the second key: fifty ship, 49 are on the revenue list too, and the lists differ by Rs 40."
#   c) "Rank Retail-Core with DENSE_RANK on orders, so that every member who ordered the same shares a place on the list."
#   d) "Keep the revenue list, since ranking by orders would drop the members whose quarters carry the most revenue."
LINE = __TODO7__
print("gap:", GAP, "| line:", LINE)

In [ ]:
carried_gap = abs(sum(r["q2_revenue"] for r in freq_list) - sum(r["q2_revenue"] for r in rev_list))
kit.check("the gap you chose is the gap the two lists carry", {"a": 40, "b": 2950, "c": 120000, "d": 0}.get(GAP) == carried_gap)
kit.check("the line is chosen", LINE in "abcd")
kit.check_summary()